In [3]:
import pandas as pd
df = pd.read_csv('data/processed/features.csv')

In [5]:
df.head(5)

,area,year,quarter,latitude,longitude,avg_d_mbps,avg_u_mbps,avg_lat_ms,tests,nearest_tower_distance_km,...,tower_count_5km,digital_elevation_model,region_enc,typeOfArea_enc,city_enc,demand_growth_pct,region,typeOfArea,city,block_number
0,A'ali,2020,1,26.167764,50.516785,34.608000,6.629200,21.200000,52,0.910574,...,263.800000,10.0,2,5,0,0.000000,Northern Governorate,PPLX,A'ali,738
1,A'ali,2020,2,26.168586,50.516052,24.785500,8.958833,23.166667,105,0.949577,...,266.166667,10.0,2,5,0,0.000000,Northern Governorate,PPLX,A'ali,714
2,A'ali,2020,3,26.168586,50.516052,28.455333,11.872833,25.000000,72,0.949577,...,266.166667,10.0,2,5,0,0.000000,Northern Governorate,PPLX,A'ali,714
3,A'ali,2020,4,26.168586,50.516052,51.100167,10.033667,26.833333,137,0.949577,...,266.166667,10.0,2,5,0,0.000000,Northern Governorate,PPLX,A'ali,714
4,A'ali,2021,1,26.168586,50.516052,50.347500,9.390667,44.500000,80,0.949577,...,266.166667,10.0,2,5,0,-0.199454,Northern Governorate,PPLX,A'ali,714


In [7]:
df.shape

(6086, 22)

In [10]:
df.dtypes

area                          object
year                           int64
quarter                        int64
latitude                     float64
longitude                    float64
avg_d_mbps                   float64
avg_u_mbps                   float64
avg_lat_ms                   float64
tests                          int64
nearest_tower_distance_km    float64
tower_count_1km              float64
tower_count_2km              float64
tower_count_5km              float64
digital_elevation_model      float64
region_enc                     int64
typeOfArea_enc                 int64
city_enc                       int64
demand_growth_pct            float64
region                        object
typeOfArea                    object
city                          object
block_number                   int64
dtype: object

In [13]:
df.columns

Index(['area', 'year', 'quarter', 'latitude', 'longitude', 'avg_d_mbps',
       'avg_u_mbps', 'avg_lat_ms', 'tests', 'nearest_tower_distance_km',
       'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
       'digital_elevation_model', 'region_enc', 'typeOfArea_enc', 'city_enc',
       'demand_growth_pct', 'region', 'typeOfArea', 'city', 'block_number'],
      dtype='object')

In [60]:
features = ['year', 'quarter', 'latitude', 'longitude', 'nearest_tower_distance_km',
       'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
       'digital_elevation_model', 'region_enc', 'typeOfArea_enc', 'city_enc',
       'demand_growth_pct']
target = 'avg_d_mbps'

In [62]:
features

['year',
 'quarter',
 'latitude',
 'longitude',
 'nearest_tower_distance_km',
 'tower_count_1km',
 'tower_count_2km',
 'tower_count_5km',
 'digital_elevation_model',
 'region_enc',
 'typeOfArea_enc',
 'city_enc',
 'demand_growth_pct']

In [63]:
df_clean = df.dropna(subset=features + [target]).copy()
len(df_clean)

6086

In [64]:
df_clean[target].min()

np.float64(0.349)

In [42]:
df_clean[target].max()

np.float64(1330.731)

In [43]:
df_clean[target].mean()

np.float64(152.42066227997978)

In [44]:
df_clean[target].median()

np.float64(131.25038461538463)

In [66]:
X = df_clean[features].values
y = df_clean[target].values

In [67]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold

In [68]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
len(X_train)

4868

In [69]:
len(X_test)

1218

In [71]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score

rf = RandomForestRegressor(
    n_estimators  = 300,
    max_depth     = 15,
    min_samples_leaf = 4,
    max_features  = 'sqrt',
    random_state  = 42,
    n_jobs        = -1,
)
rf.fit(X_train, y_train)

,n_estimators,300
,criterion,'squared_error'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,4
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [72]:
y_pred_train = rf.predict(X_train)
y_pred_test  = rf.predict(X_test)

In [78]:
def report(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"  RMSE : {rmse:8.2f} Mbps")
    print(f"  MAE  : {mae:8.2f} Mbps")
    print(f"  R²   : {r2:8.4f}")
    return rmse, mae, r2

train_rmse, train_mae, train_r2 = report(y_train, y_pred_train, "TRAIN SET")
test_rmse,  test_mae,  test_r2  = report(y_test,  y_pred_test,  "TEST SET  (held-out)")

  RMSE :    57.85 Mbps
  MAE  :    35.58 Mbps
  R²   :   0.7289
  RMSE :    80.88 Mbps
  MAE  :    49.92 Mbps
  R²   :   0.5423


In [80]:
overfit_gap = train_r2 - test_r2
overfit_gap

0.18657392988695243

In [115]:
print("\nRunning 5-fold cross-validation on training set…")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train,
                             cv=kf,
                             scoring='neg_root_mean_squared_error',
                             n_jobs=-1)
cv_rmse = -cv_scores
print(f"  CV RMSE per fold: {[f'{v:.2f}' for v in cv_rmse]}")
print(f"  CV RMSE  mean ± std: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f} Mbps")
print("  (stable std → model generalises well across different data subsets)")


Running 5-fold cross-validation on training set…
  CV RMSE per fold: ['76.34', '67.19', '83.91', '74.02', '72.65']
  CV RMSE  mean ± std: 74.82 ± 5.45 Mbps
  (stable std → model generalises well across different data subsets)


In [85]:
speeds = pd.read_csv('data/processed/speeds_clean.csv')

In [86]:
speeds.columns

Index(['date', 'latitude', 'longitude', 'avg_d_mbps', 'avg_u_mbps',
       'avg_lat_ms', 'tests', 'block_number', 'city', 'area', 'typeOfArea',
       'region', 'digital_elevation_model', 'last_modified_date', 'year',
       'quarter', 'nearest_tower_distance_km', 'tower_count_1km',
       'tower_count_2km', 'tower_count_5km', 'region_enc', 'typeOfArea_enc',
       'city_enc', 'demand_growth_pct_x', 'demand_growth_pct_y',
       'demand_growth_pct'],
      dtype='object')

In [88]:
features2 = ['year',
 'quarter',
 'latitude',
 'longitude',
 'nearest_tower_distance_km',
 'tower_count_1km',
 'tower_count_2km',
 'tower_count_5km',
 'digital_elevation_model',
 'region_enc',
 'typeOfArea_enc',
 'city_enc',
 'demand_growth_pct']
target2 = 'avg_d_mbps'

In [89]:
speeds[target].min()

np.float64(0.005)

In [90]:
speeds[target].max()

np.float64(2012.325)

In [91]:
X2 = speeds[features2].values
y2 = speeds[target2].values

In [92]:
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.20, random_state=42)

In [96]:
len(X2_train)

32020

In [95]:
len(X2_test)

8005

In [108]:
rf2 = RandomForestRegressor(
    n_estimators  = 300,
    max_depth     = 15,
    min_samples_leaf = 4,
    max_features  = 'sqrt',
    random_state  = 42,
    n_jobs        = -1,
)
rf2.fit(X2_train, y2_train)

,n_estimators,300
,criterion,'squared_error'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,4
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [109]:
y2_pred_train = rf2.predict(X2_train)
y2_pred_test  = rf2.predict(X2_test)

train2_rmse, train2_mae, train2_r2 = report(y2_train, y2_pred_train, "TRAIN SET")
test2_rmse,  test2_mae,  test2_r2  = report(y2_test,  y2_pred_test,  "TEST SET  (held-out)")


── TRAIN SET ──────────────────────────────
  RMSE :   101.75 Mbps  (avg error magnitude, penalises big misses)
  MAE  :    67.21 Mbps  (avg absolute error, more intuitive)
  R²   :   0.5518        (1.0 = perfect; 0.0 = no better than mean)

── TEST SET  (held-out) ──────────────────────────────
  RMSE :   114.52 Mbps  (avg error magnitude, penalises big misses)
  MAE  :    75.12 Mbps  (avg absolute error, more intuitive)
  R²   :   0.4446        (1.0 = perfect; 0.0 = no better than mean)


In [110]:
overfit_gap2 = train2_r2 - test2_r2

In [111]:
overfit_gap2

0.1072706334135698

In [114]:
kf2 = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores2 = cross_val_score(rf2, X2_train, y2_train,
                             cv=kf2,
                             scoring='neg_root_mean_squared_error',
                             n_jobs=-1)
cv_rmse2 = -cv_scores2
print(f"  CV RMSE per fold: {[f'{v:.2f}' for v in cv_rmse2]}")
print(f"  CV RMSE  mean ± std: {cv_rmse2.mean():.2f} ± {cv_rmse2.std():.2f} Mbps")
print("  (stable std → model generalises well across different data subsets)")

  CV RMSE per fold: ['119.54', '120.72', '118.52', '111.37', '115.24']
  CV RMSE  mean ± std: 117.08 ± 3.38 Mbps
  (stable std → model generalises well across different data subsets)


In [ ]:
  RMSE :    57.85 Mbps
  MAE  :    35.58 Mbps
  R²   :   0.7289
  RMSE :    80.88 Mbps
  MAE  :    49.92 Mbps
  R²   :   0.5423

In [7]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [8]:
df = pd.read_csv("data/processed/speeds_clean.csv")

In [9]:
df.head(5)

,date,latitude,longitude,avg_d_mbps,avg_u_mbps,avg_lat_ms,tests,block_number,city,area,...,tower_count_1km,tower_count_2km,tower_count_5km,region_enc,typeOfArea_enc,city_enc,demand_growth_pct_x,demand_growth_pct_y,demand_growth_pct,visible_tower_count_5km
0,2020-01-01,26.286027,50.578308,4.029,3.417,22,1,228,Al Busaytin,Jazirat as Sayah,...,0,3,245,1,3,11,0.0,0.0,0.0,172
1,2020-01-01,26.310651,50.622253,11.932,3.027,22,1,263,Khasaifa Island,Khasaifa Island,...,0,3,73,1,1,101,0.0,0.0,0.0,51
2,2020-01-01,26.305726,50.622253,32.232,17.256,28,87,263,Khasaifa Island,Khasaifa Island,...,1,3,101,1,1,101,0.0,0.0,0.0,66
3,2020-01-01,26.286027,50.600281,23.765,17.882,18,2,228,Al Busaytin,Jazirat as Sayah,...,3,5,312,1,3,11,0.0,0.0,0.0,227
4,2020-01-01,26.281102,50.583801,29.366,4.510,22,1,228,Al Busaytin,Jazirat as Sayah,...,1,3,432,1,3,11,0.0,0.0,0.0,299


In [10]:
df.columns

Index(['date', 'latitude', 'longitude', 'avg_d_mbps', 'avg_u_mbps',
       'avg_lat_ms', 'tests', 'block_number', 'city', 'area', 'typeOfArea',
       'region', 'digital_elevation_model', 'last_modified_date', 'year',
       'quarter', 'nearest_tower_distance_km', 'tower_count_1km',
       'tower_count_2km', 'tower_count_5km', 'region_enc', 'typeOfArea_enc',
       'city_enc', 'demand_growth_pct_x', 'demand_growth_pct_y',
       'demand_growth_pct', 'visible_tower_count_5km'],
      dtype='object')

In [11]:
df.shape

(40025, 27)

In [13]:
features = [
    "latitude",
    "longitude",
    "year",
    "quarter",
    "block_number",
    "city",
    "area",
    "typeOfArea",
    "region",
    "digital_elevation_model",
    "tests",
    "nearest_tower_distance_km",
    "tower_count_1km",
    "tower_count_2km",
    "tower_count_5km",
    "visible_tower_count_5km"
] 

In [14]:
targets = ["avg_d_mbps", "avg_u_mbps", "avg_lat_ms"]

In [15]:
for c in df.columns:
    if c.startswith("tower_count_5km"):
        print(c)

tower_count_5km


In [16]:
features = list(dict.fromkeys([c for c in features if c in df.columns]))

In [17]:
features

['latitude',
 'longitude',
 'year',
 'quarter',
 'block_number',
 'city',
 'area',
 'typeOfArea',
 'region',
 'digital_elevation_model',
 'tests',
 'nearest_tower_distance_km',
 'tower_count_1km',
 'tower_count_2km',
 'tower_count_5km',
 'visible_tower_count_5km']

In [48]:
for col in targets:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

print("After capping at p99:")
print(df[targets].describe())

After capping at p99:
         avg_d_mbps    avg_u_mbps    avg_lat_ms
count  40025.000000  40025.000000  40025.000000
mean     156.036650     22.336702     22.757421
std      144.640744     16.568901     20.390994
min        0.005000      0.001000      3.000000
25%       44.210000     10.681000     15.000000
50%      115.894000     18.985000     18.000000
75%      226.492000     29.791000     22.000000
max      718.028360     88.219360    163.760000


In [49]:
X = df[features].copy()
y = df[targets].copy()

In [20]:
X.shape

(40025, 16)

In [21]:
y

,avg_d_mbps,avg_u_mbps,avg_lat_ms
0,4.029,3.417,22
1,11.932,3.027,22
2,32.232,17.256,28
3,23.765,17.882,18
4,29.366,4.510,22
...,...,...,...
40020,24.987,12.808,20
40021,103.508,25.592,21
40022,57.552,43.786,19
40023,190.958,12.248,21


In [50]:
X.dtypes

latitude                     float64
longitude                    float64
year                           int64
quarter                        int64
block_number                   int64
city                          object
area                          object
typeOfArea                    object
region                        object
digital_elevation_model        int64
tests                          int64
nearest_tower_distance_km    float64
tower_count_1km                int64
tower_count_2km                int64
tower_count_5km                int64
visible_tower_count_5km        int64
dtype: object

In [51]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

In [24]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [25]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [27]:
#Splitting data based on the time

In [28]:
df.groupby(['year']).year.value_counts()

year
2020     5186
2021     5188
2022     6439
2023     6201
2024    12221
2025     4790
Name: count, dtype: int64

In [52]:
train_split = (df["year"] < 2024) | ((df["year"] == 2024) & (df["quarter"] <= 2))
val_split = (df["year"] == 2024) & (df["quarter"] >= 3)
test_split = (df["year"] == 2025)

In [53]:
X_train = X.loc[train_split]
X_val   = X.loc[val_split]
X_test  = X.loc[test_split] 
Y_train = y.loc[train_split]
Y_val   = y.loc[val_split]
Y_test  = y.loc[test_split]

In [32]:
train_split.sum()

np.int64(25544)

In [33]:
df.loc[train_split,'date'].min()

'2020-01-01'

In [34]:
df.loc[train_split,'date'].max()

'2024-04-01'

In [35]:
train_split

0         True
1         True
2         True
3         True
4         True
         ...  
40020    False
40021    False
40022    False
40023    False
40024    False
Length: 40025, dtype: bool

In [36]:
df.loc[val_split,'date'].min()

'2024-07-01'

In [37]:
df.loc[val_split,'date'].max()

'2024-10-01'

In [38]:
df.loc[test_split,'date'].min()

'2025-01-01'

In [39]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

In [40]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [41]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [42]:
def eval_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    """Return MAE, RMSE, and R² as a dict of Python floats."""
    return {
        "MAE":  float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2":   float(r2_score(y_true, y_pred)),
    }

In [78]:
#Models

In [43]:
lr = LinearRegression()

In [44]:
rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1,
    )

In [45]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1,
    ),
}

In [54]:
results      = {}   # {model_name: {target: {"val": metrics, "test": metrics}}}
saved_models = {}   # {model_name: {target: fitted_pipeline}}
 
for model_name, estimator in models.items():
    print(f"\n{'='*55}")
    print(f"  Model: {model_name}")
    print(f"{'='*55}")
 
    results[model_name]      = {}
    saved_models[model_name] = {}
    for target in targets:
        # --- build & fit pipeline on TRAIN only ---
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model",        estimator),
        ])
        pipe.fit(X_train, Y_train[target])
 
        # --- evaluate on VALIDATION (used for tuning / early stopping) ---
        val_preds  = pipe.predict(X_val)
        val_metrics  = eval_metrics(Y_val[target],  val_preds)
 
        results[model_name][target] = {
            "val":  val_metrics,
        }
        saved_models[model_name][target] = pipe
 
        print(f"\n  Target : {target}")
        print(f"    Validation  → MAE={val_metrics['MAE']:.3f}  "
              f"RMSE={val_metrics['RMSE']:.3f}  R²={val_metrics['R2']:.4f}")


  Model: linear_regression

  Target : avg_d_mbps
    Validation  → MAE=113.172  RMSE=143.969  R²=0.0463

  Target : avg_u_mbps
    Validation  → MAE=12.996  RMSE=16.623  R²=0.0193

  Target : avg_lat_ms
    Validation  → MAE=10.008  RMSE=19.972  R²=0.1644

  Model: random_forest

  Target : avg_d_mbps
    Validation  → MAE=101.384  RMSE=141.840  R²=0.0743

  Target : avg_u_mbps
    Validation  → MAE=11.777  RMSE=16.237  R²=0.0643

  Target : avg_lat_ms
    Validation  → MAE=10.577  RMSE=21.293  R²=0.0503


In [55]:
print(df[targets].describe())
print(df[targets].quantile([0.90, 0.95, 0.99]))

         avg_d_mbps    avg_u_mbps    avg_lat_ms
count  40025.000000  40025.000000  40025.000000
mean     156.036650     22.336702     22.757421
std      144.640744     16.568901     20.390994
min        0.005000      0.001000      3.000000
25%       44.210000     10.681000     15.000000
50%      115.894000     18.985000     18.000000
75%      226.492000     29.791000     22.000000
max      718.028360     88.219360    163.760000
      avg_d_mbps  avg_u_mbps  avg_lat_ms
0.90  346.363800   43.360000     32.0000
0.95  446.590400   53.986000     48.0000
0.99  718.012674   88.212794    163.5776


In [56]:
df_agg = df.groupby(["area","region","typeOfArea","year","quarter"]).agg(
    avg_d_mbps   = ("avg_d_mbps", "mean"),
    avg_u_mbps   = ("avg_u_mbps", "mean"),
    avg_lat_ms   = ("avg_lat_ms", "mean"),
    test_count   = ("tests", "sum"),
    n_records    = ("avg_d_mbps", "count"),
).reset_index()

In [57]:
df_agg

,area,region,typeOfArea,year,quarter,avg_d_mbps,avg_u_mbps,avg_lat_ms,test_count,n_records
0,A'ali,Northern Governorate,PPLX,2020,1,34.608000,6.629200,21.200000,52,5
1,A'ali,Northern Governorate,PPLX,2020,2,24.785500,8.958833,23.166667,105,6
2,A'ali,Northern Governorate,PPLX,2020,3,28.455333,11.872833,25.000000,72,6
3,A'ali,Northern Governorate,PPLX,2020,4,51.100167,10.033667,26.833333,137,6
4,A'ali,Northern Governorate,PPLX,2021,1,50.347500,9.390667,44.500000,80,6
...,...,...,...,...,...,...,...,...,...,...
6590,`Ayn ad Dar,Capital Governorate,PPL,2024,4,69.348000,27.279000,20.000000,12,1
6591,`Ayn ad Dar,Capital Governorate,PPL,2025,1,233.291000,34.829000,26.000000,37,1
6592,`Ayn ad Dar,Capital Governorate,PPL,2025,2,199.800000,27.873000,18.000000,37,1
6593,`Ayn ad Dar,Capital Governorate,PPL,2025,3,150.852000,16.813000,16.000000,14,1


In [58]:
df_agg = df_agg.sort_values(["area","year","quarter"])
for t in targets:
    df_agg[f"prev_q_{t}"] = df_agg.groupby("area")[t].shift(1)

In [59]:
df_agg

,area,region,typeOfArea,year,quarter,avg_d_mbps,avg_u_mbps,avg_lat_ms,test_count,n_records,prev_q_avg_d_mbps,prev_q_avg_u_mbps,prev_q_avg_lat_ms
0,A'ali,Northern Governorate,PPLX,2020,1,34.608000,6.629200,21.200000,52,5,NaN,NaN,NaN
1,A'ali,Northern Governorate,PPLX,2020,2,24.785500,8.958833,23.166667,105,6,34.608000,6.629200,21.200000
2,A'ali,Northern Governorate,PPLX,2020,3,28.455333,11.872833,25.000000,72,6,24.785500,8.958833,23.166667
3,A'ali,Northern Governorate,PPLX,2020,4,51.100167,10.033667,26.833333,137,6,28.455333,11.872833,25.000000
4,A'ali,Northern Governorate,PPLX,2021,1,50.347500,9.390667,44.500000,80,6,51.100167,10.033667,26.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6590,`Ayn ad Dar,Capital Governorate,PPL,2024,4,69.348000,27.279000,20.000000,12,1,111.154000,17.634000,22.000000
6591,`Ayn ad Dar,Capital Governorate,PPL,2025,1,233.291000,34.829000,26.000000,37,1,69.348000,27.279000,20.000000
6592,`Ayn ad Dar,Capital Governorate,PPL,2025,2,199.800000,27.873000,18.000000,37,1,233.291000,34.829000,26.000000
6593,`Ayn ad Dar,Capital Governorate,PPL,2025,3,150.852000,16.813000,16.000000,14,1,199.800000,27.873000,18.000000


In [60]:
# ── 1. Aggregate to area-quarter level ───────────────────
df_agg = df.groupby(["area","city","region","typeOfArea","year","quarter"]).agg(
    avg_d_mbps = ("avg_d_mbps","mean"),
    avg_u_mbps = ("avg_u_mbps","mean"),
    avg_lat_ms = ("avg_lat_ms","mean"),
    test_count = ("tests","sum"),
).reset_index()

# ── 2. Add lag features (now meaningful on stable aggregates) ─
df_agg = df_agg.sort_values(["area","year","quarter"]).reset_index(drop=True)
for t in targets:
    df_agg[f"prev_q_{t}"]    = df_agg.groupby("area")[t].shift(1)
    df_agg[f"rolling3_{t}"]  = df_agg.groupby("area")[t].transform(
                                    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
                                )

# ── 3. Same split logic, same masks ──────────────────────
train_mask = (df_agg["year"] < 2024) | ((df_agg["year"] == 2024) & (df_agg["quarter"] <= 2))
val_mask   = (df_agg["year"] == 2024) & (df_agg["quarter"] >= 3)
test_mask  = (df_agg["year"] == 2025)

# ── 4. Updated feature list ───────────────────────────────
numeric_features = [
    "year", "quarter", "test_count",
    "prev_q_avg_d_mbps", "prev_q_avg_u_mbps", "prev_q_avg_lat_ms",
    "rolling3_avg_d_mbps", "rolling3_avg_u_mbps", "rolling3_avg_lat_ms",
]
categorical_features = ["area", "city", "region", "typeOfArea"]
feature_cols = numeric_features + categorical_features

X = df_agg[feature_cols]
Y = df_agg[targets]

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
Y_train, Y_val, Y_test = Y[train_mask], Y[val_mask], Y[test_mask]


In [62]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [63]:
X = df_agg[feature_cols]
Y = df_agg[targets]

X_train = X[train_mask]
X_val   = X[val_mask]
X_test  = X[test_mask]
Y_train = Y[train_mask]
Y_val   = Y[val_mask]
Y_test  = Y[test_mask]

In [64]:
# Quick sanity check — will raise immediately if any column is missing
print("All features present:", all(c in df_agg.columns for c in feature_cols))
print("Missing:", [c for c in feature_cols if c not in df_agg.columns])

All features present: True
Missing: []


In [65]:
results      = {}   # {model_name: {target: {"val": metrics, "test": metrics}}}
saved_models = {}   # {model_name: {target: fitted_pipeline}}
 
for model_name, estimator in models.items():
    print(f"\n{'='*55}")
    print(f"  Model: {model_name}")
    print(f"{'='*55}")
 
    results[model_name]      = {}
    saved_models[model_name] = {}
    for target in targets:
        # --- build & fit pipeline on TRAIN only ---
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model",        estimator),
        ])
        pipe.fit(X_train, Y_train[target])
 
        # --- evaluate on VALIDATION (used for tuning / early stopping) ---
        val_preds  = pipe.predict(X_val)
        val_metrics  = eval_metrics(Y_val[target],  val_preds)
 
        results[model_name][target] = {
            "val":  val_metrics,
        }
        saved_models[model_name][target] = pipe
 
        print(f"\n  Target : {target}")
        print(f"    Validation  → MAE={val_metrics['MAE']:.3f}  "
              f"RMSE={val_metrics['RMSE']:.3f}  R²={val_metrics['R2']:.4f}")


  Model: linear_regression

  Target : avg_d_mbps
    Validation  → MAE=75.620  RMSE=98.939  R²=0.1186

  Target : avg_u_mbps
    Validation  → MAE=8.937  RMSE=11.843  R²=0.1771

  Target : avg_lat_ms
    Validation  → MAE=9.573  RMSE=16.699  R²=0.3146

  Model: random_forest

  Target : avg_d_mbps
    Validation  → MAE=67.729  RMSE=96.612  R²=0.1595

  Target : avg_u_mbps
    Validation  → MAE=8.728  RMSE=12.253  R²=0.1192

  Target : avg_lat_ms
    Validation  → MAE=9.717  RMSE=17.056  R²=0.2850


In [66]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=400,
        random_state=42,
        min_samples_leaf=3,
        n_jobs=-1,
    ),
}

In [67]:
results      = {}   # {model_name: {target: {"val": metrics, "test": metrics}}}
saved_models = {}   # {model_name: {target: fitted_pipeline}}
 
for model_name, estimator in models.items():
    print(f"\n{'='*55}")
    print(f"  Model: {model_name}")
    print(f"{'='*55}")
 
    results[model_name]      = {}
    saved_models[model_name] = {}
    for target in targets:
        # --- build & fit pipeline on TRAIN only ---
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model",        estimator),
        ])
        pipe.fit(X_train, Y_train[target])
 
        # --- evaluate on VALIDATION (used for tuning / early stopping) ---
        val_preds  = pipe.predict(X_val)
        val_metrics  = eval_metrics(Y_val[target],  val_preds)
 
        results[model_name][target] = {
            "val":  val_metrics,
        }
        saved_models[model_name][target] = pipe
 
        print(f"\n  Target : {target}")
        print(f"    Validation  → MAE={val_metrics['MAE']:.3f}  "
              f"RMSE={val_metrics['RMSE']:.3f}  R²={val_metrics['R2']:.4f}")


  Model: linear_regression

  Target : avg_d_mbps
    Validation  → MAE=75.620  RMSE=98.939  R²=0.1186

  Target : avg_u_mbps
    Validation  → MAE=8.937  RMSE=11.843  R²=0.1771

  Target : avg_lat_ms
    Validation  → MAE=9.573  RMSE=16.699  R²=0.3146

  Model: random_forest

  Target : avg_d_mbps
    Validation  → MAE=67.658  RMSE=96.637  R²=0.1591

  Target : avg_u_mbps
    Validation  → MAE=8.736  RMSE=12.270  R²=0.1168

  Target : avg_lat_ms
    Validation  → MAE=9.641  RMSE=17.024  R²=0.2877


In [ ]:

# Time-based split
train_mask = df["year"] < 2025
test_mask = df["year"] == 2025

if test_mask.sum() == 0:
    train_mask = np.arange(len(df)) % 5 != 0
    test_mask = ~train_mask

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
Y_train = Y.loc[train_mask]
Y_test = Y.loc[test_mask]

def eval_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ),
}

results = {}
saved_models = {}

for model_name, estimator in models.items():
    results[model_name] = {}
    saved_models[model_name] = {}

    for target in targets:
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", estimator)
        ])

        pipe.fit(X_train, Y_train[target])
        preds = pipe.predict(X_test)

        results[model_name][target] = eval_metrics(Y_test[target], preds)
        saved_models[model_name][target] = pipe

# Save metrics
metrics_df = []
for model_name, target_dict in results.items():
    for target, metric_dict in target_dict.items():
        row = {"model": model_name, "target": target}
        row.update(metric_dict)
        metrics_df.append(row)

metrics_df = pd.DataFrame(metrics_df)
metrics_df.to_csv("outputs/model_metrics.csv", index=False)

# Save model bundle
reference_points = df[[
    "latitude", "longitude", "block_number", "city", "area",
    "typeOfArea", "region", "digital_elevation_model", "tests"
]].drop_duplicates().reset_index(drop=True)

towers = pd.read_csv("outputs/towers_clean.csv")

bundle = {
    "models": saved_models,
    "metrics": results,
    "feature_columns": feature_cols,
    "reference_points": reference_points,
    "towers": towers
}

joblib.dump(bundle, "models/model_bundle.joblib")

print("Saved:")
print("- models/model_bundle.joblib")
print("- outputs/model_metrics.csv")
print("\nMetrics:")
print(metrics_df)

In [1]:
import os, warnings, numpy as np, pandas as pd, joblib
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

df     = pd.read_csv("data/processed/speeds_clean.csv")
towers = pd.read_csv("outputs/towers_clean.csv")

In [2]:
targets = ["avg_d_mbps", "avg_u_mbps", "avg_lat_ms"]

for col in targets:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

In [11]:
df_agg = df.groupby(["area","region","typeOfArea","year","quarter"]).agg(
    avg_d_mbps = ("avg_d_mbps", "mean"),
    avg_u_mbps = ("avg_u_mbps", "mean"),
    avg_lat_ms = ("avg_lat_ms", "mean"),
    test_count = ("tests",      "sum"),
).reset_index()

print(f"Aggregated rows: {len(df_agg)}")  # should be a few hundred

Aggregated rows: 6595


In [12]:
df_agg = df_agg.sort_values(["area","year","quarter"]).reset_index(drop=True)

for t in targets:
    df_agg[f"prev_q_{t}"]   = df_agg.groupby("area")[t].shift(1)
    df_agg[f"rolling3_{t}"] = df_agg.groupby("area")[t].transform(
                                  lambda x: x.shift(1).rolling(3, min_periods=1).mean()
                              )

In [13]:
train_mask = (df_agg["year"] < 2024) | ((df_agg["year"] == 2024) & (df_agg["quarter"] <= 2))
val_mask   = (df_agg["year"] == 2024) & (df_agg["quarter"] >= 3)
test_mask  = (df_agg["year"] == 2025)

print(f"Train: {train_mask.sum()} | Val: {val_mask.sum()} | Test: {test_mask.sum()}")

Train: 4969 | Val: 546 | Test: 1080


In [15]:
numeric_features = [
    "year", "quarter", "test_count",
    "prev_q_avg_d_mbps", "prev_q_avg_u_mbps", "prev_q_avg_lat_ms",
    "rolling3_avg_d_mbps", "rolling3_avg_u_mbps", "rolling3_avg_lat_ms",
]
categorical_features = ["area", "region", "typeOfArea"]
feature_cols = numeric_features + categorical_features

X = df_agg[feature_cols]
Y = df_agg[targets]

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
Y_train, Y_val, Y_test = Y[train_mask], Y[val_mask], Y[test_mask]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]),              numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_features),
])

In [29]:
def eval_metrics(y_true, y_pred):
    return {
        "MAE":  float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2":   float(r2_score(y_true, y_pred)),
    }

models = {
    "linear_regression": LinearRegression(),
    "random_forest":     RandomForestRegressor(n_estimators=300, random_state=42,
                                               min_samples_leaf=2, n_jobs=-1),
}

results, saved_models = {}, {}

for model_name, estimator in models.items():
    print(f"\n{'='*50}\n  Model: {model_name}\n{'='*50}")
    results[model_name], saved_models[model_name] = {}, {}

    for target in targets:
        pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
        pipe.fit(X_train, Y_train[target])

        val_preds  = pipe.predict(X_val)

        results[model_name][target] = {
            "val":  eval_metrics(Y_val[target],  val_preds)
        }
        saved_models[model_name][target] = pipe

        v = results[model_name][target]["val"]
        print(f"\n  Target : {target}")
        print(f"    Validation  → MAE={v['MAE']:.3f}  RMSE={v['RMSE']:.3f}  R²={v['R2']:.4f}")


  Model: linear_regression

  Target : avg_d_mbps
    Validation  → MAE=71.991  RMSE=92.396  R²=0.0919

  Target : avg_u_mbps
    Validation  → MAE=8.238  RMSE=10.917  R²=0.1685

  Target : avg_lat_ms
    Validation  → MAE=9.320  RMSE=15.802  R²=0.3810

  Model: random_forest

  Target : avg_d_mbps
    Validation  → MAE=63.359  RMSE=88.383  R²=0.1691

  Target : avg_u_mbps
    Validation  → MAE=7.885  RMSE=10.958  R²=0.1623

  Target : avg_lat_ms
    Validation  → MAE=9.237  RMSE=16.020  R²=0.3638


In [17]:
print("df_agg shape:", df_agg.shape)
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Columns in X_train:", X_train.columns.tolist())

df_agg shape: (6595, 15)
X_train shape: (4969, 12)
X_val shape: (546, 12)
X_test shape: (1080, 12)
Columns in X_train: ['year', 'quarter', 'test_count', 'prev_q_avg_d_mbps', 'prev_q_avg_u_mbps', 'prev_q_avg_lat_ms', 'rolling3_avg_d_mbps', 'rolling3_avg_u_mbps', 'rolling3_avg_lat_ms', 'area', 'region', 'typeOfArea']


In [18]:
print("Unique areas:", df["area"].nunique())
print("Unique year-quarters:", (df["year"].astype(str) + "-Q" + df["quarter"].astype(str)).nunique())
print("Expected agg rows ≈", df["area"].nunique() * (df["year"].astype(str) + "-Q" + df["quarter"].astype(str)).nunique())
print("\ndf_agg sample:")
print(df_agg.groupby("area").size().describe())

Unique areas: 273
Unique year-quarters: 24
Expected agg rows ≈ 6552

df_agg sample:
count    273.000000
mean      24.157509
std        8.467114
min        1.000000
25%       24.000000
50%       24.000000
75%       24.000000
max       53.000000
dtype: float64


In [19]:
print(f"df_agg shape: {df_agg.shape}")

df_agg shape: (6595, 15)


In [20]:
df_agg = df.groupby(["area", "region", "typeOfArea", "year", "quarter"]).agg(
    avg_d_mbps = ("avg_d_mbps", "mean"),
    avg_u_mbps = ("avg_u_mbps", "mean"),
    avg_lat_ms = ("avg_lat_ms", "mean"),
    test_count = ("tests",      "sum"),
).reset_index()

# Step 2 — rejoin city using the most common city per area
city_map = df.groupby("area")["city"].agg(lambda x: x.mode()[0])
df_agg["city"] = df_agg["area"].map(city_map)

print(f"df_agg shape: {df_agg.shape}")  # should be ≤ 6,552

df_agg shape: (6595, 10)


In [24]:
df_agg

,area,region,typeOfArea,year,quarter,avg_d_mbps,avg_u_mbps,avg_lat_ms,test_count,city
0,A'ali,Northern Governorate,PPLX,2020,1,34.608000,6.629200,21.200000,52,A'ali
1,A'ali,Northern Governorate,PPLX,2020,2,24.785500,8.958833,23.166667,105,A'ali
2,A'ali,Northern Governorate,PPLX,2020,3,28.455333,11.872833,25.000000,72,A'ali
3,A'ali,Northern Governorate,PPLX,2020,4,51.100167,10.033667,26.833333,137,A'ali
4,A'ali,Northern Governorate,PPLX,2021,1,50.347500,9.390667,44.500000,80,A'ali
...,...,...,...,...,...,...,...,...,...,...
6590,`Ayn ad Dar,Capital Governorate,PPL,2024,4,69.348000,27.279000,20.000000,12,`Ayn ad Dar
6591,`Ayn ad Dar,Capital Governorate,PPL,2025,1,233.291000,34.829000,26.000000,37,`Ayn ad Dar
6592,`Ayn ad Dar,Capital Governorate,PPL,2025,2,199.800000,27.873000,18.000000,37,`Ayn ad Dar
6593,`Ayn ad Dar,Capital Governorate,PPL,2025,3,150.852000,16.813000,16.000000,14,`Ayn ad Dar


In [27]:
df_agg = df_agg.sort_values(["area","year","quarter"])
for t in targets:
    df_agg[f"prev_q_{t}"] = df_agg.groupby("area")[t].shift(1)
    
print("Lag feature NaN rate:")
for t in targets:
    pct = df_agg[f"prev_q_{t}"].isna().mean() * 100
    print(f"  prev_q_{t}: {pct:.1f}% NaN")

print("\nAreas with only 1 quarter of data:", 
      (df_agg.groupby("area").size() == 1).sum())

print("\nAreas with 4+ quarters:", 
      (df_agg.groupby("area").size() >= 4).sum())

print("\nCorrelation of lag features with targets:")
for t in targets:
    corr = df_agg[f"prev_q_{t}"].corr(df_agg[t])
    print(f"  prev_q_{t} → {t}: {corr:.4f}")

Lag feature NaN rate:
  prev_q_avg_d_mbps: 4.1% NaN
  prev_q_avg_u_mbps: 4.1% NaN
  prev_q_avg_lat_ms: 4.1% NaN

Areas with only 1 quarter of data: 7

Areas with 4+ quarters: 262

Correlation of lag features with targets:
  prev_q_avg_d_mbps → avg_d_mbps: 0.6057
  prev_q_avg_u_mbps → avg_u_mbps: 0.5197
  prev_q_avg_lat_ms → avg_lat_ms: 0.3594


In [28]:
df_agg

,area,region,typeOfArea,year,quarter,avg_d_mbps,avg_u_mbps,avg_lat_ms,test_count,city,prev_q_avg_d_mbps,prev_q_avg_u_mbps,prev_q_avg_lat_ms
0,A'ali,Northern Governorate,PPLX,2020,1,34.608000,6.629200,21.200000,52,A'ali,NaN,NaN,NaN
1,A'ali,Northern Governorate,PPLX,2020,2,24.785500,8.958833,23.166667,105,A'ali,34.608000,6.629200,21.200000
2,A'ali,Northern Governorate,PPLX,2020,3,28.455333,11.872833,25.000000,72,A'ali,24.785500,8.958833,23.166667
3,A'ali,Northern Governorate,PPLX,2020,4,51.100167,10.033667,26.833333,137,A'ali,28.455333,11.872833,25.000000
4,A'ali,Northern Governorate,PPLX,2021,1,50.347500,9.390667,44.500000,80,A'ali,51.100167,10.033667,26.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6590,`Ayn ad Dar,Capital Governorate,PPL,2024,4,69.348000,27.279000,20.000000,12,`Ayn ad Dar,111.154000,17.634000,22.000000
6591,`Ayn ad Dar,Capital Governorate,PPL,2025,1,233.291000,34.829000,26.000000,37,`Ayn ad Dar,69.348000,27.279000,20.000000
6592,`Ayn ad Dar,Capital Governorate,PPL,2025,2,199.800000,27.873000,18.000000,37,`Ayn ad Dar,233.291000,34.829000,26.000000
6593,`Ayn ad Dar,Capital Governorate,PPL,2025,3,150.852000,16.813000,16.000000,14,`Ayn ad Dar,199.800000,27.873000,18.000000


In [32]:
rf_pipe = saved_models["random_forest"]["avg_d_mbps"]
pre = rf_pipe.named_steps["preprocessor"]
ohe_cols = pre.transformers_[1][1].named_steps["ohe"].get_feature_names_out(categorical_features).tolist()
all_feat = numeric_features + ohe_cols
importances = rf_pipe.named_steps["model"].feature_importances_
top10 = sorted(zip(all_feat, importances), key=lambda x: -x[1])[:10]
for name, imp in top10:
    print(f"  {name:40s} {imp:.4f}")

  rolling3_avg_lat_ms                      0.1715
  area_Al Jufayr                           0.1179
  test_count                               0.0839
  prev_q_avg_lat_ms                        0.0677
  prev_q_avg_u_mbps                        0.0661
  rolling3_avg_u_mbps                      0.0622
  rolling3_avg_d_mbps                      0.0619
  prev_q_avg_d_mbps                        0.0596
  quarter                                  0.0330
  year                                     0.0284


In [33]:
def eval_metrics(y_true, y_pred):
    return {
        "MAE":  float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2":   float(r2_score(y_true, y_pred)),
    }

models = {
    "linear_regression": LinearRegression(),
    "random_forest":     RandomForestRegressor(n_estimators=300, random_state=42,
                                               min_samples_leaf=2, n_jobs=-1),
}

results, saved_models = {}, {}

for model_name, estimator in models.items():
    print(f"\n{'='*50}\n  Model: {model_name}\n{'='*50}")
    results[model_name], saved_models[model_name] = {}, {}

    for target in targets:
        pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
        pipe.fit(X_train, Y_train[target])

        val_preds  = pipe.predict(X_val)

        results[model_name][target] = {
            "val":  eval_metrics(Y_val[target],  val_preds)
        }
        saved_models[model_name][target] = pipe

        v = results[model_name][target]["val"]
        print(f"\n  Target : {target}")
        print(f"    Validation  → MAE={v['MAE']:.3f}  RMSE={v['RMSE']:.3f}  R²={v['R2']:.4f}")


  Model: linear_regression

  Target : avg_d_mbps
    Validation  → MAE=71.991  RMSE=92.396  R²=0.0919

  Target : avg_u_mbps
    Validation  → MAE=8.238  RMSE=10.917  R²=0.1685

  Target : avg_lat_ms
    Validation  → MAE=9.320  RMSE=15.802  R²=0.3810

  Model: random_forest

  Target : avg_d_mbps
    Validation  → MAE=63.359  RMSE=88.383  R²=0.1691

  Target : avg_u_mbps
    Validation  → MAE=7.885  RMSE=10.958  R²=0.1623

  Target : avg_lat_ms
    Validation  → MAE=9.237  RMSE=16.020  R²=0.3638
